<a href="https://colab.research.google.com/github/GokulM8/Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: [quote or paraphrase the claim, with a page/section reference]**

- *Where does the label come from?* [What outcome is being measured, and how — self-reported? measured directly? proxied by something else?]
- *Does the validation design carry the claim?* [What split or comparison backs this number — is it grouped/time-aware, or could the same site/client appear in both the group the claim is about and the baseline it's compared against?]
- *Constructive note:* [One respectful sentence — what would make this finding easier to trust as-is.]

**Finding 2: [quote or paraphrase the claim, with a page/section reference]**

- *Where does the label come from?* [...]
- *Does the validation design carry the claim?* [...]
- *Constructive note:* [...]

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Setup — same warehouse connection and features as ML-08

In [1]:
%pip install -q duckdb huggingface_hub

In [2]:
from google.colab import userdata
from huggingface_hub import HfApi
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

In [3]:
api = HfApi()
all_files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
config_files = sorted(f for f in all_files if "fact_content_daily_performance" in f and f.endswith(".parquet"))

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{HF_TOKEN}')""")

remote_paths = [f"hf://datasets/FlyRank/internship-warehouse/{f}" for f in config_files]
paths_sql = "[" + ", ".join(f"'{p}'" for p in remote_paths) + "]"
con.sql(f"CREATE OR REPLACE VIEW fact AS SELECT * FROM read_parquet({paths_sql})")

In [4]:
march_features = con.sql("""
    WITH march AS (
        SELECT *, CAST(strftime(report_date, '%d') AS INTEGER) AS day,
            sessions_organic + sessions_direct + sessions_referral
            + sessions_social + sessions_paid + sessions_ai AS total_sessions_row
        FROM fact
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    )
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_impressions) AS total_impressions,
        SUM(sessions_ai) * 1.0 / NULLIF(SUM(total_sessions_row), 0) AS ai_search_share,
        SUM(ga4_engaged_sessions) * 1.0 / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,
        SUM(gsc_clicks) AS march_total_clicks
    FROM march
    GROUP BY client_hash_id, content_hash_id
""").df().fillna(0)

april_totals = con.sql("""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS april_total_clicks
    FROM fact
    WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
    GROUP BY client_hash_id, content_hash_id
""").df()

joined = march_features.merge(april_totals, on=["client_hash_id", "content_hash_id"], how="inner")
joined["is_declining_forward"] = (joined["april_total_clicks"] < joined["march_total_clicks"]).astype(int)

feature_cols = ["avg_ctr", "avg_position", "total_impressions", "ai_search_share", "engagement_rate"]
X = joined[feature_cols].fillna(0)
y = joined["is_declining_forward"]
groups = joined["client_hash_id"]

print("Rows:", len(joined), " Clients:", groups.nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 331436  Clients: 55


### BEFORE — naive random split (no grouping)

A client's pages can land on both sides here — this is the split most people reach for by default.

In [5]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model_random = RandomForestClassifier(
    n_estimators=300, max_depth=5, min_samples_leaf=10, class_weight="balanced", random_state=42,
)
model_random.fit(X_train_r, y_train_r)
auc_random = roc_auc_score(y_test_r, model_random.predict_proba(X_test_r)[:, 1])

print(f"ROC-AUC, naive random split: {auc_random:.3f}")

ROC-AUC, naive random split: 0.964


### AFTER — grouped split by client (`GroupShuffleSplit`)

Every client's pages stay entirely on one side. This is what ML-08 actually used.

In [6]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

model_grouped = RandomForestClassifier(
    n_estimators=300, max_depth=5, min_samples_leaf=10, class_weight="balanced", random_state=42,
)
model_grouped.fit(X_train_g, y_train_g)
auc_grouped = roc_auc_score(y_test_g, model_grouped.predict_proba(X_test_g)[:, 1])

print(f"ROC-AUC, grouped-by-client split: {auc_grouped:.3f}")
print(f"Gap (random - grouped): {auc_random - auc_grouped:+.3f}")
print("A positive gap here means the naive split was quietly optimistic — some of that score was the")
print("model recognizing a client it had already seen in train, not a genuine content signal.")

ROC-AUC, grouped-by-client split: 0.952
Gap (random - grouped): +0.012
A positive gap here means the naive split was quietly optimistic — some of that score was the
model recognizing a client it had already seen in train, not a genuine content signal.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [9]:
# Step 1 — confirm every model input is March-only, nothing from April/June
print("Features feeding the model:", feature_cols)
print("All five are aggregated from fact_content_daily_performance, March 2026 rows only.")
print("april_total_clicks is used ONLY to build the label — it is not in feature_cols.")

Features feeding the model: ['avg_ctr', 'avg_position', 'total_impressions', 'ai_search_share', 'engagement_rate']
All five are aggregated from fact_content_daily_performance, March 2026 rows only.
april_total_clicks is used ONLY to build the label — it is not in feature_cols.


In [11]:
# Step 2 — the trap, repeated on the real label: inject an April-derived column on purpose
# and watch the score jump toward perfect, exactly like the w03 leakage exercise.
leaky_cols = feature_cols + ["march_total_clicks"]  # still March-only, but nearly defines the label by construction

X_leak = joined[leaky_cols].fillna(0)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak, y, test_size=0.3, random_state=42, stratify=y
)
model_leak = RandomForestClassifier(
    n_estimators=300, max_depth=5, min_samples_leaf=10, class_weight="balanced", random_state=42,
)
model_leak.fit(X_train_l, y_train_l)
auc_leak = roc_auc_score(y_test_l, model_leak.predict_proba(X_test_l)[:, 1])

print(f"ROC-AUC with march_total_clicks added: {auc_leak:.3f}")
print(f"ROC-AUC, honest 5 features only (grouped split): {auc_grouped:.3f}")
print(f"Jump: {auc_leak - auc_grouped:+.3f}")
print("march_total_clicks isn't the label, but it's so close to it structurally (the label just compares")
print("march_total_clicks to april_total_clicks) that including it lets the model partially back into the")
print("answer. It stays OUT of feature_cols for exactly this reason.")

ROC-AUC with march_total_clicks added: 0.968
ROC-AUC, honest 5 features only (grouped split): 0.952
Jump: +0.016
march_total_clicks isn't the label, but it's so close to it structurally (the label just compares
march_total_clicks to april_total_clicks) that including it lets the model partially back into the
answer. It stays OUT of feature_cols for exactly this reason.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (Week-5 submission report):**
> "[Model/baseline] performed better."

**Rewritten:**
> On this March→April slice, under a client-grouped split, the model showed a directionally higher ROC-AUC and Precision@10 than the Week-4 rule — an observed, decision-support signal on one month of data, not a general claim that the model beats the rule. The gap between the naive-split and grouped-split AUC above (section 2) also means even this number should be read as measured on a specific held-out slice, not treated as a stable, guaranteed margin.

## Self-check

Before you submit, confirm each line honestly:

- [-] Every section above is filled — markdown thinking AND the code that backs it
- [-] The notebook runs top to bottom with no errors (Runtime → Run all)
- [-] No client names, URLs, or private queries anywhere
- [-] My claims use careful words: observed, measured, directional, decision-support
- [-] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.